## 手動でのコンテンツベースフィルタリング

このラボでは、低レベルの TensorFlow オペレーションを使用してコンテンツベースフィルタを実装する方法を説明します。  
ここでのコードは、「Recommendation Engines: Content Based Filtering」のモジュール 2 で説明されている手法に従っています。

## 学習目標
1. ユーザー特徴量行列の背後にある数学を理解する
2. ユーザー特徴量とアイテム特徴量からユーザー評価を計算する方法を理解する

In [ ]:
import tensorflow as tf

まず、ユーザー、映画、特徴量のリストを作成します。ユーザーと映画はデータベース内の要素を表しますが、コンテンツベースフィルタリング手法では、映画の特徴量は多くの場合手作業でエンジニアリングされ、最適な埋め込み空間を提供するためにドメイン知識に依存します。ここでは、Action、Sci-Fi、Comedy、Cartoon、Drama のカテゴリを使用して、映画（およびユーザー）を記述します。

この例では、データベースが以下に示す 4 人のユーザーと 6 本の映画で構成されていると仮定します。

In [ ]:
users = ["Ryan", "Danielle", "Vijay", "Chris"]
movies = [
    "Star Wars",
    "The Dark Knight",
    "Shrek",
    "The Incredibles",
    "Bleu",
    "Memento",
]
features = ["Action", "Sci-Fi", "Comedy", "Cartoon", "Drama"]

num_users = len(users)
num_movies = len(movies)
num_feats = len(features)
num_recommendations = 2

### ユーザー、映画の評価、特徴量の初期化

ユーザーの映画評価と、k-hot エンコードされた映画特徴量行列を入力する必要があります。`users_movies` 行列の各行は、各映画に対する単一のユーザーの評価（1 から 10）を表します。0 は、ユーザーがその映画を視聴/評価していないことを示します。`movies_feats` 行列には、指定された各映画の特徴量が含まれます。各行は 6 本の映画のいずれかを表し、列は 5 つのカテゴリを表します。1 は、映画が特定のジャンル/カテゴリに該当することを示します。

In [ ]:
# 各行は、異なる映画に対するユーザーの評価を表します。
users_movies = tf.constant(
    [
        [4, 6, 8, 0, 0, 0],
        [0, 0, 10, 0, 8, 3],
        [0, 6, 0, 0, 3, 7],
        [10, 9, 0, 5, 0, 2],
    ],
    dtype=tf.float32,
)

# 映画の特徴量をワンホット（k-hot）エンコードしたもの。
# 例: 各列は以下を表す
# ['Action', 'Sci-Fi', 'Comedy', 'Cartoon', 'Drama']
movies_feats = tf.constant(
    [
        [1, 1, 0, 0, 1],
        [1, 1, 0, 0, 0],
        [0, 0, 1, 1, 0],
        [1, 0, 1, 1, 0],
        [0, 0, 0, 0, 1],
        [1, 0, 0, 0, 1],
    ],
    dtype=tf.float32,
)

### ユーザー特徴量行列の計算

ユーザー特徴量行列、つまり 5 次元の特徴量空間における各ユーザーの埋め込みを含む行列を計算します。

**TODO 1**: これを `users_movies` テンソルと `movies_feats` テンソルの行列積として計算します。

In [ ]:
users_feats = tf.matmul(users_movies, movies_feats)
users_feats

次に、各ユーザー特徴量ベクトルの合計が 1 になるように正規化します。正規化は厳密には必須ではありませんが、ユーザー間で評価の大きさを比較できるようにします。

In [ ]:
users_feats = users_feats / tf.reduce_sum(users_feats, axis=1, keepdims=True)
users_feats

#### 各ユーザーの特徴量の関連性のランキング

上記で計算した `users_feats` を使用して、各ユーザーに対する各映画カテゴリの相対的な重要度を表すことができます。

In [ ]:
top_users_features = tf.nn.top_k(users_feats, num_feats)[1]
top_users_features

In [ ]:
for i in range(num_users):
    feature_names = [features[int(index)] for index in top_users_features[i]]
    print(f"{users[i]}: {feature_names}")

### おすすめ映画の決定

上記で計算した `users_feats` テンソルを使用して、各ユーザーに対する映画の評価とおすすめを決定します。

各映画の予測評価を計算するために、ユーザーの特徴量ベクトルと対応する映画の特徴量ベクトル間の類似度尺度を計算します。  

類似度尺度としてドット積（内積）を使用します。本質的には、これは各ユーザーに対する重み付けされた映画の平均です。

**TODO 2**: これを行列積として実装します。*ヒント:* オペランドの 1 つを転置する必要があります。

In [ ]:
users_ratings = tf.matmul(users_feats, tf.transpose(movies_feats))
users_ratings

上記の計算により、データベース内の各ユーザーと各映画の間の類似度尺度が算出されます。新しい映画の評価のみに焦点を当てるために、`users_ratings` 行列にマスクを適用します。  

ユーザーがすでに映画を評価している場合、その評価は無視します。これにより、これまでに視聴/評価されていない映画の評価のみに注目します。

In [ ]:
users_unseen_movies = tf.equal(users_movies, tf.zeros_like(users_movies))
ignore_matrix = tf.zeros_like(tf.cast(users_movies, tf.float32))

users_ratings_new = tf.where(users_unseen_movies, users_ratings, ignore_matrix)

users_ratings_new

最後に、各ユーザーに対して評価の高い上位 2 つの映画を取得して出力しましょう。

In [ ]:
top_movies = tf.nn.top_k(users_ratings_new, num_recommendations)[1]
top_movies

In [ ]:
for i in range(num_users):
    movie_names = [movies[index] for index in top_movies[i]]
    print(f"{users[i]}: {movie_names}")

Copyright 2019 Google Inc.
Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at
http://www.apache.org/licenses/LICENSE-2.0
Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License.